# 04 - SAO (synchronous single-GPU approximation)

The paper's SAO system decouples asynchronous rollout workers from training. This one-GPU notebook does **not** claim to reproduce that system: it synchronously collects one solution per problem, performs two critic updates, recomputes values, then applies one DIS policy update. Because the rollout policy and current policy are initially identical in this approximation, the first update usually masks few or no tokens; the metric is still measured rather than fabricated.

In [ ]:
# Hyperparameters
sft_checkpoint = "./checkpoints/sft_base"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 256
eval_problems = 64
num_epochs = 1
rollout_batch_size = 128
generation_micro_batch_size = 16
forward_micro_batch_size = 8
max_prompt_tokens = 384
max_completion_tokens = 256
temperature = 0.8
top_p = 1.0
actor_learning_rate = 1e-6
critic_learning_rate = 5e-6
weight_decay = 0.0
epsilon_low = 0.8
epsilon_high = 3.0
gamma = 1.0
gae_lambda = 0.95
critic_updates_per_policy_update = 2
critic_warmstart_problems = 128
max_grad_norm = 1.0
eval_every_steps = 1
eval_batch_size = 16
wandb_project = "swe-rl-ablations"
wandb_run_name = "sao"
output_path = "./checkpoints/sao"

optimizer_steps = train_problems // rollout_batch_size
assert rollout_batch_size == 128 and critic_updates_per_policy_update == 2
assert epsilon_low == 0.8 and epsilon_high == 3.0
assert critic_warmstart_problems % rollout_batch_size == 0
assert train_problems == 256 and eval_problems == 64 and num_epochs == 1
print(f"SAO updates: {optimizer_steps}; critic updates/policy update: {critic_updates_per_policy_update}")

In [ ]:
from pathlib import Path
import os
import re
import time

import torch
import wandb
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, set_seed

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = project_root / sft_checkpoint
if not (checkpoint_path / "config.json").exists():
    raise FileNotFoundError("Run 00_sft.ipynb first.")
os.environ["WANDB_PROJECT"] = wandb_project
set_seed(seed)

train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision).shuffle(seed=seed).select(range(train_problems))
eval_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision).shuffle(seed=seed).select(range(eval_problems))

In [ ]:
def gold_number(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K answer: {answer!r}")
    return match.group(1).replace(",", "")


def predicted_number(completion):
    match = re.search(r"Final answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", completion.strip(), re.IGNORECASE)
    return None if not match else match.group(1).replace(",", "")


def exact_reward(completion, answer):
    prediction = predicted_number(completion)
    if prediction is None:
        return -1.0
    return 1.0 if float(prediction) == float(gold_number(answer)) else 0.0


def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum().clamp_min(1)


def gradient_norm(parameters):
    norms = [parameter.grad.detach().norm(2) for parameter in parameters if parameter.grad is not None]
    return float(torch.norm(torch.stack(norms), 2).item()) if norms else 0.0


def explained_variance(prediction, target, mask):
    prediction = prediction[mask.bool()]
    target = target[mask.bool()]
    variance = target.var(unbiased=False)
    return 0.0 if variance <= 1e-12 else float((1 - (target - prediction).var(unbiased=False) / variance).item())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def prompt_text(question):
    messages = [{"role": "user", "content": f"Solve this grade-school math problem. Show your reasoning, then end with `Final answer: <number>`.\n\n{question}"}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

actor = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2").to("cuda")
reference = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2").eval().to("cuda")
critic = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_path,
    num_labels=1,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).to("cuda")
critic.config.pad_token_id = tokenizer.pad_token_id
for parameter in reference.parameters():
    parameter.requires_grad_(False)

trainable_critic_parameters = []
for name, parameter in critic.named_parameters():
    parameter.requires_grad_("mlp" in name.lower() or "score" in name.lower() or "classifier" in name.lower())
    if parameter.requires_grad:
        trainable_critic_parameters.append(name)
if not trainable_critic_parameters:
    raise RuntimeError("No dense MLP or scalar-head critic parameters were enabled.")
if any("attn" in name.lower() for name, parameter in critic.named_parameters() if parameter.requires_grad):
    raise RuntimeError("Attention parameters must remain frozen in the SAO critic.")
print(f"Trainable critic tensors: {len(trainable_critic_parameters)}")

In [ ]:
def generate_rollout(model, rows, sample=True):
    input_ids_parts = []
    attention_parts = []
    completion_mask_parts = []
    completion_text = []
    prompts = [prompt_text(question) for question in rows["question"]]
    for start in range(0, len(prompts), generation_micro_batch_size):
        chunk = prompts[start:start + generation_micro_batch_size]
        encoded = tokenizer(chunk, return_tensors="pt", padding="max_length", truncation=True, max_length=max_prompt_tokens).to(model.device)
        generated = model.generate(
            **encoded,
            do_sample=sample,
            temperature=temperature if sample else None,
            top_p=top_p if sample else None,
            max_new_tokens=max_completion_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        completion = generated[:, max_prompt_tokens:]
        if completion.shape[1] < max_completion_tokens:
            padding = torch.full((len(chunk), max_completion_tokens - completion.shape[1]), tokenizer.pad_token_id, device=model.device)
            completion = torch.cat([completion, padding], 1)
        eos = completion.eq(tokenizer.eos_token_id)
        first_eos = torch.where(eos.any(1), eos.float().argmax(1), torch.full((len(chunk),), max_completion_tokens - 1, device=model.device)).long()
        positions = torch.arange(max_completion_tokens, device=model.device).unsqueeze(0)
        generated_mask = positions <= first_eos.unsqueeze(1)
        ids = torch.cat([encoded.input_ids, completion], 1)
        mask = torch.cat([encoded.attention_mask, generated_mask.long()], 1)
        completion_mask = torch.zeros((len(chunk), ids.shape[1] - 1), device=model.device)
        completion_mask[:, max_prompt_tokens - 1:] = generated_mask.float()
        input_ids_parts.append(ids)
        attention_parts.append(mask)
        completion_mask_parts.append(completion_mask)
        completion_text.extend(tokenizer.batch_decode(completion, skip_special_tokens=True))
    return torch.cat(input_ids_parts), torch.cat(attention_parts), torch.cat(completion_mask_parts), completion_text


def token_logprobs(model, input_ids, attention_mask, requires_grad):
    parts = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_micro_batch_size):
            ids = input_ids[start:start + forward_micro_batch_size]
            mask = attention_mask[start:start + forward_micro_batch_size]
            logits = model(input_ids=ids, attention_mask=mask, use_cache=False).logits[:, :-1]
            parts.append(torch.log_softmax(logits.float(), -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1))
    return torch.cat(parts)


def token_values(model, input_ids, attention_mask, requires_grad):
    parts = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_micro_batch_size):
            ids = input_ids[start:start + forward_micro_batch_size]
            mask = attention_mask[start:start + forward_micro_batch_size]
            output = model(input_ids=ids, attention_mask=mask, output_hidden_states=True, use_cache=False, return_dict=True)
            parts.append(model.score(output.hidden_states[-1][:, :-1]).squeeze(-1).float())
    return torch.cat(parts)


def monte_carlo_targets(reward, mask):
    return reward.unsqueeze(1).expand_as(mask) * mask


def full_gae(reward, values, mask):
    token_rewards = torch.zeros_like(values)
    final_positions = (torch.arange(mask.shape[1], device=mask.device).unsqueeze(0) * mask.long()).max(1).values
    token_rewards[torch.arange(len(reward), device=mask.device), final_positions] = reward
    advantages = torch.zeros_like(values)
    running = torch.zeros(len(values), device=values.device)
    for index in range(values.shape[1] - 1, -1, -1):
        next_value = values[:, index + 1] * mask[:, index + 1] if index + 1 < values.shape[1] else 0.0
        delta = token_rewards[:, index] + gamma * next_value - values[:, index]
        running = (delta + gamma * gae_lambda * running) * mask[:, index]
        advantages[:, index] = running
    selected = advantages[mask.bool()]
    advantages = (advantages - selected.mean()) / (selected.std(unbiased=False) + 1e-8)
    return advantages * mask

In [ ]:
@torch.no_grad()
def evaluate_exact():
    actor.eval()
    rewards = []
    for start in range(0, len(eval_rows), eval_batch_size):
        stop = min(start + eval_batch_size, len(eval_rows))
        rows = eval_rows.select(range(start, stop))
        _, _, _, completions = generate_rollout(actor, rows, sample=False)
        rewards.extend(exact_reward(completion, answer) for completion, answer in zip(completions, rows["answer"]))
    actor.train()
    return sum(rewards) / len(rewards)

actor_optimizer = torch.optim.AdamW(actor.parameters(), lr=actor_learning_rate, weight_decay=weight_decay)
critic_optimizer = torch.optim.AdamW(
    (parameter for parameter in critic.parameters() if parameter.requires_grad),
    lr=critic_learning_rate,
    weight_decay=weight_decay,
)
run = wandb.init(project=wandb_project, name=wandb_run_name)
torch.cuda.reset_peak_memory_stats()

Warm-start the newly initialized scalar value head before changing the policy. With a terminal GSM8K reward and `gamma=1`, the Monte Carlo value target is the final reward at every valid generated token.

In [ ]:
warmstart_step = 0
for start in range(0, critic_warmstart_problems, rollout_batch_size):
    rows = train_rows.select(range(start, start + rollout_batch_size))
    with torch.no_grad():
        input_ids, attention_mask, completion_mask, completions = generate_rollout(actor, rows)
        reward = torch.tensor([exact_reward(completion, answer) for completion, answer in zip(completions, rows["answer"])], device=actor.device)
        targets = monte_carlo_targets(reward, completion_mask)
    started_at = time.perf_counter()
    values = token_values(critic, input_ids, attention_mask, requires_grad=True)
    loss = masked_mean((values - targets).square(), completion_mask)
    critic_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    norm = gradient_norm(critic.parameters())
    torch.nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
    critic_optimizer.step()
    warmstart_step += 1
    run.log({
        "warmstart/critic_loss": float(loss.item()),
        "warmstart/grad_norm_critic": norm,
        "warmstart/reward": float(reward.mean().item()),
        "warmstart/time_critic_forward_backward_s": time.perf_counter() - started_at,
        "gpu/peak_memory_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }, step=warmstart_step)

In [ ]:
global_step = 0
for start in range(0, len(train_rows), rollout_batch_size):
    step_started = time.perf_counter()
    rows = train_rows.select(range(start, start + rollout_batch_size))
    with torch.no_grad():
        input_ids, attention_mask, completion_mask, completions = generate_rollout(actor, rows)
        reward = torch.tensor([exact_reward(completion, answer) for completion, answer in zip(completions, rows["answer"])], device=actor.device)
        rollout_logps = token_logprobs(actor, input_ids, attention_mask, requires_grad=False)
        reference_logps = token_logprobs(reference, input_ids, attention_mask, requires_grad=False)
        mc_targets = monte_carlo_targets(reward, completion_mask)

    critic_losses = []
    critic_norms = []
    critic_seconds = 0.0
    for _ in range(critic_updates_per_policy_update):
        torch.cuda.synchronize()
        started_at = time.perf_counter()
        values = token_values(critic, input_ids, attention_mask, requires_grad=True)
        critic_loss = masked_mean((values - mc_targets).square(), completion_mask)
        critic_optimizer.zero_grad(set_to_none=True)
        critic_loss.backward()
        critic_norms.append(gradient_norm(critic.parameters()))
        torch.nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
        critic_optimizer.step()
        torch.cuda.synchronize()
        critic_seconds += time.perf_counter() - started_at
        critic_losses.append(float(critic_loss.item()))

    with torch.no_grad():
        current_values = token_values(critic, input_ids, attention_mask, requires_grad=False)
        advantages = full_gae(reward, current_values, completion_mask)

    torch.cuda.synchronize()
    started_at = time.perf_counter()
    current_logps = token_logprobs(actor, input_ids, attention_mask, requires_grad=True)
    ratio = torch.exp(current_logps - rollout_logps)
    active = ((ratio > 1 - epsilon_low) & (ratio < 1 + epsilon_high)).float()
    policy_loss = -masked_mean(ratio * advantages * active, completion_mask)
    actor_optimizer.zero_grad(set_to_none=True)
    policy_loss.backward()
    actor_norm = gradient_norm(actor.parameters())
    torch.nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
    actor_optimizer.step()
    torch.cuda.synchronize()
    policy_seconds = time.perf_counter() - started_at

    global_step += 1
    metrics = {
        "train/reward": float(reward.mean().item()),
        "loss/policy": float(policy_loss.item()),
        "objective/kl_sft": float(masked_mean(rollout_logps - reference_logps, completion_mask).item()),
        "policy/tokens_clipped_or_masked_pct": 100 * float(masked_mean(1 - active, completion_mask).item()),
        "critic/loss": sum(critic_losses) / len(critic_losses),
        "critic/explained_variance": explained_variance(current_values, mc_targets, completion_mask),
        "grad_norm/policy": actor_norm,
        "grad_norm/critic": sum(critic_norms) / len(critic_norms),
        "time/policy_forward_backward_s": policy_seconds,
        "time/critic_forward_backward_s": critic_seconds,
        "time/step_s": time.perf_counter() - step_started,
        "gpu/peak_memory_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }
    if global_step % eval_every_steps == 0 or global_step == optimizer_steps:
        metrics["eval/reward"] = evaluate_exact()
    run.log(metrics, step=warmstart_step + global_step)

actor.save_pretrained(project_root / output_path)
tokenizer.save_pretrained(project_root / output_path)
run.finish()